# NLP Forward

Use a nonlinear equality constraint with labeled data.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from kkthn import KKTHardNet

TRAIN = {
    "epochs": 2,
    "batch_size": 8,
    "learning_rate": 1e-3,
    "train_frac": 0.8,
    "hidden_size": 16,
    "hidden_layers": 1,
    "seed": 42,
    "dtype": "float64",
    "print_every": 1,
}


## Generate Example Data

In [ ]:
data_dir = Path('notebooks/data/nlp_forward')
data_dir.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(2)
y1 = rng.uniform(0.0, 1.0, size=32)
y2 = rng.uniform(0.0, 0.5, size=32)
x1 = np.sin(y1) + y2
x2 = y2

pd.DataFrame({'x1': x1, 'x2': x2}).to_csv(data_dir / 'parameters.csv', index=False)
pd.DataFrame({'y1': y1, 'y2': y2}).to_csv(data_dir / 'variables.csv', index=False)
data_dir


## Build And Train

In [ ]:
model = KKTHardNet(name='nlp_forward', train=TRAIN)
x = model.add_parameter(['x1', 'x2'])
y = model.add_variable(['y1', 'y2'])
model.constraints.add(
    model.sin(y.y1) + y.y2 - x.x1 == 0,
    y.y2 - x.x2 == 0,
    y.y1 >= 0,
)
model.dataset(parameters=data_dir / 'parameters.csv', variables=data_dir / 'variables.csv')
result = model.model()
result['metadata_path']


In [ ]:
KKTHardNet().load(result['metadata_path']).predict([0.9, 0.2])
